### Deterministic Approach

In [3]:
import re

# Deterministic approach
def deterministic_guardrail(text: str) -> bool:
    """Returns true if content is blocked"""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do i hack into a database?",
    "What is the capital of france?",
    "Explain how malware spreads"
]

print("== Deterministic Guardrails Demo ==")
for input in test_inputs:
    blocked = deterministic_guardrail(input)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {input}")

== Deterministic Guardrails Demo ==
BLOCKED: How do i hack into a database?
ALLOWED: What is the capital of france?
BLOCKED: Explain how malware spreads


### Model Based Approach

In [12]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

def model_based_guardrails(text: str) -> str:
    """Uses LLM to evaluate content safety, Returns SAFE or UNSAFE"""
    model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)
    prompt = f"""Is the following user input safe to process ? Reply with only 'SAFE' or 'UNSAFE'.
    
    Input: {text}"""

    result = model.invoke(prompt)
    return result.content.strip()

test_inputs = [
    "How do i hack into a database?",
    "What is the capital of france?",
    "Explain how malware spreads?",
]

print("== Model-Based Guardrails Demo ==")
for input in test_inputs:
    verdict = model_based_guardrails(input)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {input}")


== Model-Based Guardrails Demo ==
UNSAFE: How do i hack into a database?
SAFE: What is the capital of france?
SAFE: Explain how malware spreads?


In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool 

@tool 
def customer_lookup(query: str) -> str:
    """Look up customer info"""
    return f"Customer record found for query: {query}"


# create agent with PII middleware
agent = create_agent(
    model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0),
    tools = [customer_lookup],
    middleware=[PIIMiddleware(
        "email",
        strategy="redact",
        apply_to_input=True,
    ),
    PIIMiddleware(
        "credit_card",
        strategy="mask",
        apply_to_input=True,
    ),
    PIIMiddleware(
        "api_key",
        detector=r"(sk_|gsk_)[a-zA-Z0-9]{20,}",
        strategy="block",
        apply_to_input=True,
    )
    ],
)

print("Agent with PII middleware created successfully")

Agent with PII middleware created successfully


In [18]:
## Test PII Redaction

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("== Agent Response ==")
print(result["messages"][-1].content)

== Agent Response ==
I can't provide direct access to customer information. However, I can guide you on how to look up your customer information. Would you like that?


In [16]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='6f6795e9-c643-4b1c-affe-8cdc89a18263'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'crx4pgxh9', 'function': {'arguments': '{"query":"[REDACTED_EMAIL] ****-****-****-5100"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 731, 'total_tokens': 773, 'completion_time': 0.095698123, 'completion_tokens_details': None, 'prompt_time': 0.019463584, 'prompt_tokens_details': None, 'queue_time': 0.045206836, 'total_time': 0.115161707}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cce46-a5ff-72d2-a5f3-7e56db21c037-0', tool_calls=[{'name': 'customer_lookup', 'args

In [23]:
## Test API Key blocking

try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: YOUR_API_KEY"
        }]
    })

except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


### Human in Loop Guardrails

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command 
from langchain_core.tools import tool 


@tool
def search_web(query: str) -> str:
    """Search the web for information"""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient"""
    return f"Email sent to {to} with subject: {subject}"


@tool 
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database"""
    return f"Deleted records from {table} where {condition}"


# Create agent with the HITL middleware
hitl_agent = create_agent(
    model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0),
    tools = [search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            },
        )
    ],

    checkpointer = InMemorySaver(),
)

print("Human in the loop agent created")

Human in the loop agent created


In [27]:
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]}, config=config
)

print("== Agent paused - awaiting human approval ==")
print(result)

== Agent paused - awaiting human approval ==
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='3d790cdb-cc5b-4653-ae04-38db56580dd8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1v0tryff1', 'function': {'arguments': '{"body":"The Q4 results are available for review.","subject":"Q4 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 810, 'total_tokens': 865, 'completion_time': 0.126505796, 'completion_tokens_details': None, 'prompt_time': 0.021369174, 'prompt_tokens_details': None, 'queue_time': 0.045224516, 'total_time': 0.14787497}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ccffe-b179-7820-ace4-47

In [28]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)

print("== Approved! Final response ==")
print(approved_result["messages"][-1].content)

== Approved! Final response ==
I can't provide a response that includes a tool response. Here is the corrected response:




In [29]:
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]}, config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config = config2
)

print("== Rejected! Final response ==")
print(rejected_result["messages"][-1].content)

== Rejected! Final response ==
I can't perform this action as it requires additional functionality beyond what is available in the given functions.


### Custom Guardrails - Before Agent Hook

In [30]:
from typing import Any 
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool 

class ContentFilterMiddleware(AgentMiddleware):
    """Determinsitic guardrail: Block requests containing banned keywords"""

    def __init__(self, banned_keywords: list[str]):
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None 
        
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None 
        
        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked - keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content."
                            "Please rephrase your request"
                        )
                        
                        
                    }],
                    "jump_to": "end"
                }
            
            return None
        
@tool
def search_tool(query: str) -> str:
    """Search for information"""
    return f"Results for: {query}"


# create agent with content filter
filtered_agent = create_agent(
    model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0),
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)
print("Content filter agent created!")        


Content filter agent created!


In [31]:
result = filtered_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What is machine learning?"
    }]
})
print("Safe request response:")
print(result["messages"][-1].content)

Safe request response:
Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to perform a specific task without using explicit instructions, but rather relying on patterns and inference derived from data. It is a field that has rapidly evolved over the past few decades and has become a crucial part of many modern technologies, including image and speech recognition, natural language processing, and predictive analytics.

In traditional programming, a computer is given a set of rules and data to work with, and it follows those rules to produce a specific output. In contrast, machine learning algorithms allow computers to learn from data and improve their performance on a task over time, without being explicitly programmed.

There are several types of machine learning, including:

1. **Supervised learning**: The algorithm is trained on labeled data, where the correct output is already known. The goal is 

In [32]:
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do i hack into a server?"}]
})
print("Unsafe request response:")
print(result["messages"][-1].content)

Blocked - keyword detected: 'hack'
Unsafe request response:
I cannot process requests containing inappropriate content.Please rephrase your request
